# MedQuAD — Urology Subset EDA

MedQuAD has no specialty tag, so we filter by keyword on the `focus` field (the disease/drug/entity each QA pair is about) — matching kidney, bladder, prostate, urinary, ureter, urethra, nephro-, cystitis, incontinence, and related terms.


In [1]:
!pip install -q textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 36.1 MB/s eta 0:00:00


In [2]:
import os

REPO_DIR = "MedQuAD"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/abachaa/MedQuAD.git

source_folders = sorted([d for d in os.listdir(REPO_DIR)
                          if os.path.isdir(os.path.join(REPO_DIR, d)) and d[0].isdigit()])
print(f"{len(source_folders)} source folders found")

Cloning into 'MedQuAD'...
remote: Enumerating objects: 11291, done.
remote: Counting objects: 100% (11291/11291), done.
remote: Compressing objects: 100% (4495/4495), done.
remote: Total 11291 (delta 6796), reused 11288 (delta 6796), pack-reused 0 (from 0)
Receiving objects: 100% (11291/11291), 11.00 MiB | 11.45 MiB/s, done.
Resolving deltas: 100% (6796/6796), done.
Updating files: 100% (11277/11277), done.
12 source folders found


In [3]:
import xml.etree.ElementTree as ET
import glob
import pandas as pd

rows = []
for folder in source_folders:
    for filepath in glob.glob(os.path.join(REPO_DIR, folder, "*.xml")):
        try:
            tree = ET.parse(filepath)
            root = tree.getroot()
            focus_elem = root.find("Focus")
            focus = focus_elem.text if focus_elem is not None else None

            qapairs = root.find("QAPairs")
            if qapairs is None:
                continue

            for qa in qapairs.findall("QAPair"):
                q_elem = qa.find("Question")
                a_elem = qa.find("Answer")
                rows.append({
                    "source_folder": folder,
                    "focus": focus,
                    "qtype": q_elem.get("qtype") if q_elem is not None else None,
                    "question": q_elem.text if q_elem is not None else None,
                    "answer": a_elem.text if a_elem is not None else None,
                })
        except ET.ParseError:
            continue

full_df = pd.DataFrame(rows)
print(f"Parsed {len(full_df)} total QA pairs")

Parsed 47441 total QA pairs


## 1. Filter to urology-relevant focus terms

In [4]:
UROLOGY_PATTERN = r"kidney|bladder|prostate|urin|ureter|urethra|nephro|cystitis|incontinen|renal"

df = full_df[full_df["focus"].fillna("").str.contains(UROLOGY_PATTERN, case=False, regex=True)].copy()
print(f"Urology-relevant QA pairs: {len(df)} out of {len(full_df)} total")

df["focus"].value_counts().head(20)

Urology-relevant QA pairs: 1508 out of 47441 total


,count
focus,
Prostate Cancer,43
Kidney Disease,17
Congenital adrenal hyperplasia,15
Kidney Dysplasia,13
Nephrocalcinosis,13
Medullary cystic kidney disease,13
Urinary Tract Infections in Children,13
What I need to know about Kidney Failure and How Its Treated,12
Amyloidosis and Kidney Disease,12


## 2. Match-quality check
Which keyword in the pattern actually triggered each match — lets you spot likely false positives (e.g. 'renal' showing up in an unrelated systemic/endocrine topic) before trusting the rest of the analysis.

In [5]:
import re

keyword_list = ["kidney", "bladder", "prostate", "urin", "ureter", "urethra",
                "nephro", "cystitis", "incontinen", "renal"]

def matched_keyword(focus):
    focus_l = str(focus).lower()
    for kw in keyword_list:
        if kw in focus_l:
            return kw
    return None

df["matched_keyword"] = df["focus"].apply(matched_keyword)
df["matched_keyword"].value_counts()

,count
matched_keyword,
kidney,405
urin,315
renal,269
nephro,163
prostate,105
bladder,97
incontinen,46
cystitis,46
ureter,39


In [6]:
# Eyeball the 'renal' matches specifically — most likely source of false positives
df[df["matched_keyword"] == "renal"]["focus"].unique()

array(['Renal papillary necrosis', 'Renal venogram',
       'Cushing syndrome due to adrenal tumor', 'Renal vein thrombosis',
       'Acute adrenal crisis', 'Adrenalectomy', 'Renal scan',
       'Renal and urological disorders', 'Hepatorenal syndrome',
       'Prerenal azotemia', 'Renal', 'Renal cell carcinoma',
       'Perirenal abscess', 'Adrenal glands',
       'Renal perfusion scintiscan', 'Renal arteriography',
       'Distal renal tubular acidosis', 'Atheroembolic renal disease',
       'Distal splenorenal shunt', 'Congenital adrenal hyperplasia',
       'Proximal renal tubular acidosis', 'Renal Cell Cancer',
       'Hereditary leiomyomatosis and renal cell cancer',
       'Clear cell renal cell carcinoma',
       'Histidinuria renal tubular defect',
       'Renal dysplasia megalocystis sirenomelia',
       'Joubert syndrome with oculorenal anomalies',
       'Dandy-Walker cyst with Renal-Hepatic-Pancreatic dysplasia',
       'Oculo skeletal renal syndrome', 'Neurofaciodigitorena

## 3. Source folder breakdown within the urology subset
Which of the 12 NIH sources contribute the urology content.

In [7]:
df["source_folder"].value_counts()

,count
source_folder,
10_MPlus_ADAM_QA,659
5_NIDDK_QA,429
2_GARD_QA,130
3_GHR_QA,120
7_SeniorHealth_QA,63
1_CancerGov_QA,49
4_MPlus_Health_Topics_QA,27
11_MPlusDrugs_QA,19
12_MPlusHerbsSupplements_QA,8


## 4. Answer coverage within the urology subset

In [8]:
df["has_answer"] = df["answer"].notna() & (df["answer"].str.strip().str.len() > 0)
df["has_answer"].value_counts(normalize=True)

,proportion
has_answer,
True,0.545093
False,0.454907


## 5. Question type distribution within the urology subset

In [9]:
df["qtype"].value_counts()

,count
qtype,
information,430
treatment,168
symptoms,162
causes,110
exams and tests,107
considerations,92
outlook,71
prevention,66
when to contact a medical professional,62


## 6. Question length

In [10]:
df["question_word_count"] = df["question"].fillna("").str.split().str.len()
df["question_word_count"].describe()

,question_word_count
count,1508.000000
mean,9.045756
std,2.577510
min,4.000000
25%,7.000000
50%,9.000000
75%,11.000000
max,19.000000


## 7. Answer length — urology subset

In [11]:
answered = df[df["has_answer"]].copy()
answered["answer_word_count"] = answered["answer"].str.split().str.len()
answered["answer_word_count"].describe()

,answer_word_count
count,822.000000
mean,251.664234
std,292.195622
min,6.000000
25%,94.000000
50%,169.000000
75%,286.000000
max,3013.000000


## 8. Readability of urology answers
Comparable number to the MTSamples readability score — lets you say 'MedQuAD's urology answers score X on Flesch-Kincaid vs. MTSamples' urology reports scoring Y,' directly supporting the 'consumer Q&A is easier than clinical transcription' story.

In [12]:
import textstat

def safe_fk(text):
    text = str(text)
    return textstat.flesch_kincaid_grade(text) if len(text.split()) >= 10 else None

answered["flesch_kincaid_grade"] = answered["answer"].apply(safe_fk)
answered["flesch_kincaid_grade"].describe()

,flesch_kincaid_grade
count,821.000000
mean,12.324733
std,6.226827
min,3.689399
25%,9.126100
50%,11.531852
75%,13.303521
max,63.130968


## 9. Duplicate question check
Same question text appearing more than once (can happen across overlapping sources).

In [13]:
n_dupes = df["question"].duplicated().sum()
print(f"Duplicate questions: {n_dupes} out of {len(df)}")
df[df["question"].duplicated(keep=False)].sort_values("question")[["source_folder", "focus", "question"]].head(10)

Duplicate questions: 230 out of 1508


,source_folder,focus,question
17276,10_MPlus_ADAM_QA,Urinalysis,Do you have information about Urinalysis
43531,4_MPlus_Health_Topics_QA,Urinalysis,Do you have information about Urinalysis
44422,5_NIDDK_QA,Adrenal Insufficiency and Addison's Disease,How to diagnose Adrenal Insufficiency and Addi...
44421,5_NIDDK_QA,Adrenal Insufficiency and Addison's Disease,How to diagnose Adrenal Insufficiency and Addi...
44270,5_NIDDK_QA,Amyloidosis and Kidney Disease,How to diagnose Amyloidosis and Kidney Disease ?
44271,5_NIDDK_QA,Amyloidosis and Kidney Disease,How to diagnose Amyloidosis and Kidney Disease ?
15194,10_MPlus_ADAM_QA,Congenital adrenal hyperplasia,How to diagnose Congenital adrenal hyperplasia ?
32283,2_GARD_QA,Congenital adrenal hyperplasia,How to diagnose Congenital adrenal hyperplasia ?
43635,5_NIDDK_QA,Diabetic Kidney Disease,How to diagnose Diabetic Kidney Disease ?
43636,5_NIDDK_QA,Diabetic Kidney Disease,How to diagnose Diabetic Kidney Disease ?
